# Categorical Evidence | Issue #20

Verify the frozen Gate 2 categorical-analysis outputs and produce the Gate 3 categorical evidence required by Issue #20.

This notebook does not redefine the categorical-analysis method. It checks the frozen inventory and level-to-target tables, reproduces the required starting counts, and identifies low-support, dominant, high-cardinality, and uneven target patterns with their denominators and supports.

No categorical field or rare level is removed in this notebook.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / "data" / "allstate_claims_data.csv").exists():
            return path

    raise FileNotFoundError("Repository root not found.")


ROOT = find_repo_root()

GATE2 = (
    ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 2"
)

OUTPUT_DIR = (
    ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 3"
    / "categorical-evidence"
)

INVENTORY_PATH = GATE2 / "categorical_inventory.csv"
LEVEL_TABLE_PATH = GATE2 / "categorical_level_target_tables.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_ROWS = 188_318
RARE_THRESHOLD = 188
DOMINANT_SHARE_THRESHOLD = 0.99

print("Output directory:", OUTPUT_DIR.relative_to(ROOT))

Output directory: notebooks\final-deliverables\September\Gate 3\categorical-evidence


## Load Frozen Gate 2 Evidence

Use the categorical inventory and level-to-target tables produced by the approved categorical-analysis workflow.

The Gate 2 rare-level rule defines a rare level as one supported by fewer than 188 claims, approximately 0.1% of the 188,318-row dataset.

For this evidence review, a field is flagged as highly dominant when its most-common level represents at least 99% of all rows. This is an evidence flag only, not a removal rule.

In [3]:
inventory = pd.read_csv(INVENTORY_PATH)
level_tables = pd.read_csv(LEVEL_TABLE_PATH)

required_inventory_columns = {
    "field",
    "n_levels",
    "most_common_level",
    "most_common_support",
    "most_common_share",
    "least_common_level",
    "least_common_support",
    "min_support",
    "median_support",
    "n_rare_levels",
    "rare_row_share",
}

required_level_columns = {
    "field",
    "level",
    "support",
    "mean_loss",
    "median_loss",
    "q25_loss",
    "q75_loss",
    "iqr_loss",
    "is_rare",
}

assert required_inventory_columns.issubset(inventory.columns)
assert required_level_columns.issubset(level_tables.columns)

print(f"Inventory rows: {len(inventory)}")
print(f"Level-to-target rows: {len(level_tables)}")

Inventory rows: 116
Level-to-target rows: 1139


## 1. Reproduce Required Inventory Checks

Confirm that the inventory contains all 116 categorical fields and reproduce the cardinality checks specified in Issue #20.

In [4]:
assert inventory["field"].nunique() == 116
assert inventory["field"].is_unique

two_level_fields = int((inventory["n_levels"] == 2).sum())
four_or_fewer = int((inventory["n_levels"] <= 4).sum())

checks = {
    "categorical_fields": inventory["field"].nunique(),
    "fields_with_2_levels": two_level_fields,
    "fields_with_at_most_4_levels": four_or_fewer,
    "cat116_levels": int(
        inventory.loc[inventory["field"] == "cat116", "n_levels"].iloc[0]
    ),
    "cat110_levels": int(
        inventory.loc[inventory["field"] == "cat110", "n_levels"].iloc[0]
    ),
    "cat109_levels": int(
        inventory.loc[inventory["field"] == "cat109", "n_levels"].iloc[0]
    ),
}

assert checks["categorical_fields"] == 116
assert checks["fields_with_2_levels"] == 72
assert checks["fields_with_at_most_4_levels"] == 88
assert checks["cat116_levels"] == 326
assert checks["cat110_levels"] == 131
assert checks["cat109_levels"] == 84

summary = pd.DataFrame(
    [
        {"check": key, "observed_value": value, "status": "PASS"}
        for key, value in checks.items()
    ]
)

display(summary)

summary.to_csv(
    OUTPUT_DIR / "categorical_evidence_summary.csv",
    index=False
)

print("Required inventory checks: PASS")

,check,observed_value,status
0,categorical_fields,116,PASS
1,fields_with_2_levels,72,PASS
2,fields_with_at_most_4_levels,88,PASS
3,cat116_levels,326,PASS
4,cat110_levels,131,PASS
5,cat109_levels,84,PASS


Required inventory checks: PASS


## 2. Low-Support Evidence

Identify every level below the frozen rare-level threshold.

For every flagged level, report:

- field and level;
- support;
- denominator;
- share of the full dataset;
- raw mean loss;
- raw median loss; and
- raw loss IQR.

Rare levels remain in the evidence and are not deleted or combined.

In [5]:
low_support = level_tables[
    level_tables["support"] < RARE_THRESHOLD
].copy()

low_support["denominator"] = N_ROWS
low_support["row_share"] = (
    low_support["support"] / low_support["denominator"]
)

low_support = low_support[
    [
        "field",
        "level",
        "support",
        "denominator",
        "row_share",
        "mean_loss",
        "median_loss",
        "iqr_loss",
    ]
].sort_values(
    ["support", "field", "level"]
)

assert (low_support["support"] < RARE_THRESHOLD).all()

low_support.to_csv(
    OUTPUT_DIR / "low_support_evidence.csv",
    index=False
)

print(f"Fields containing rare levels: {(inventory['n_rare_levels'] > 0).sum()}")
print(f"Rare levels: {len(low_support)}")
print(f"Rows represented by rare levels: {low_support['support'].sum():,}")

display(low_support.head(20))

Fields containing rare levels: 39
Rare levels: 523
Rows represented by rare levels: 17,412


,field,level,support,denominator,row_share,mean_loss,median_loss,iqr_loss
318,cat101,H,1,188318,0.000005,1320.72,1320.72,0.0
319,cat101,N,1,188318,0.000005,2192.72,2192.72,0.0
320,cat101,U,1,188318,0.000005,15972.49,15972.49,0.0
328,cat102,H,1,188318,0.000005,18646.37,18646.37,0.0
329,cat102,J,1,188318,0.000005,17435.36,17435.36,0.0
378,cat105,S,1,188318,0.000005,3637.20,3637.20,0.0
379,cat105,T,1,188318,0.000005,51383.04,51383.04,0.0
504,cat109,AG,1,188318,0.000005,1150.72,1150.72,0.0
505,cat109,AK,1,188318,0.000005,544.06,544.06,0.0
507,cat109,B,1,188318,0.000005,759.84,759.84,0.0


## 3. Dominant-Level Evidence

Flag fields where the most-common level represents at least 99% of the full dataset.

For each comparison, report the dominant level, its support, the denominator, and its share.

Dominance is documented as an EDA observation. It is not an instruction to drop the field.

In [6]:
dominant = inventory[
    inventory["most_common_share"] >= DOMINANT_SHARE_THRESHOLD
].copy()

dominant["denominator"] = N_ROWS

dominant = dominant[
    [
        "field",
        "n_levels",
        "most_common_level",
        "most_common_support",
        "denominator",
        "most_common_share",
        "least_common_level",
        "least_common_support",
        "n_rare_levels",
    ]
].sort_values(
    "most_common_share",
    ascending=False
)

dominant.to_csv(
    OUTPUT_DIR / "dominant_level_evidence.csv",
    index=False
)

print(
    "Fields with a level representing at least 99% of rows:",
    len(dominant)
)

display(dominant)

Fields with a level representing at least 99% of rows: 31


,field,n_levels,most_common_level,most_common_support,denominator,most_common_share,least_common_level,least_common_support,n_rare_levels
69,cat70,2,A,188295,188318,0.999878,B,23,1
14,cat15,2,A,188284,188318,0.999819,B,34,1
21,cat22,2,A,188275,188318,0.999772,B,43,1
61,cat62,2,A,188273,188318,0.999761,B,45,1
63,cat64,2,A,188271,188318,0.999750,B,47,1
62,cat63,2,A,188239,188318,0.999580,B,79,1
67,cat68,2,A,188176,188318,0.999246,B,142,1
54,cat55,2,A,188173,188318,0.999230,B,145,1
55,cat56,2,A,188136,188318,0.999034,B,182,1
19,cat20,2,A,188114,188318,0.998917,B,204,0


## 4. Uneven Target-Distribution Evidence

Identify fields with the largest differences in raw median loss across adequately supported levels.

To avoid treating very small groups as equally reliable, only levels with support of at least 188 claims are used for this comparison.

For each field, compare the adequately supported levels with the lowest and highest raw median loss and report:

- each level;
- support;
- denominator and row share;
- raw median loss; and
- the difference in raw median loss.

The table ranks the ten largest observed spreads. It is descriptive evidence, not feature importance or evidence of causation.

In [7]:
supported = level_tables[
    level_tables["support"] >= RARE_THRESHOLD
].copy()

rows = []

for field, table in supported.groupby("field"):

    if len(table) < 2:
        continue

    low = table.loc[table["median_loss"].idxmin()]
    high = table.loc[table["median_loss"].idxmax()]

    rows.append({
        "field": field,
        "supported_levels": len(table),

        "low_level": low["level"],
        "low_support": int(low["support"]),
        "low_denominator": N_ROWS,
        "low_row_share": low["support"] / N_ROWS,
        "low_median_loss": low["median_loss"],

        "high_level": high["level"],
        "high_support": int(high["support"]),
        "high_denominator": N_ROWS,
        "high_row_share": high["support"] / N_ROWS,
        "high_median_loss": high["median_loss"],

        "median_loss_spread": (
            high["median_loss"] - low["median_loss"]
        ),
    })

uneven = (
    pd.DataFrame(rows)
    .sort_values("median_loss_spread", ascending=False)
    .head(10)
)

uneven.to_csv(
    OUTPUT_DIR / "uneven_target_evidence.csv",
    index=False
)

display(uneven)

,field,supported_levels,low_level,low_support,low_denominator,low_row_share,low_median_loss,high_level,high_support,high_denominator,high_row_share,high_median_loss,median_loss_spread
65,cat57,2,A,185296,188318,0.983953,2081.930,B,3022,188318,0.016047,9532.340,7450.410
95,cat89,3,A,183744,188318,0.975711,2073.635,B,4312,188318,0.022897,7666.925,5593.290
75,cat7,2,A,183744,188318,0.975711,2073.635,B,4574,188318,0.024289,7614.575,5540.940
3,cat101,11,A,106721,188318,0.566706,1739.250,Q,2762,188318,0.014667,6016.505,4277.255
17,cat114,11,E,16475,188318,0.087485,1308.600,U,250,188318,0.001328,5480.140,4171.540
93,cat87,4,A,788,188318,0.004184,1486.840,D,11719,188318,0.062230,5270.120,3783.280
84,cat79,4,B,152929,188318,0.812079,1858.840,D,26657,188318,0.141553,5440.810,3581.970
91,cat85,4,A,788,188318,0.004184,1486.840,D,514,188318,0.002729,4932.485,3445.645
83,cat78,4,A,788,188318,0.004184,1486.840,D,359,188318,0.001906,4927.440,3440.600
97,cat90,3,A,177993,188318,0.945173,2033.380,C,728,188318,0.003866,5240.120,3206.740


## 5. High-Cardinality Review

Review `cat116`, `cat110`, and `cat109` without assuming that high cardinality makes the fields unusable.

Report total levels, adequately supported levels, rare levels, rare-row support, and the full-dataset denominator.

Rare levels remain represented in the frozen target table.

In [8]:
HIGH_CARDINALITY_FIELDS = [
    "cat116",
    "cat110",
    "cat109",
]

rows = []

for field in HIGH_CARDINALITY_FIELDS:

    inv = inventory.loc[
        inventory["field"] == field
    ].iloc[0]

    table = level_tables[
        level_tables["field"] == field
    ]

    rare = table[
        table["support"] < RARE_THRESHOLD
    ]

    rows.append({
        "field": field,
        "total_levels": int(inv["n_levels"]),
        "supported_levels": int(
            (table["support"] >= RARE_THRESHOLD).sum()
        ),
        "rare_levels": int(len(rare)),
        "rare_rows": int(rare["support"].sum()),
        "denominator": N_ROWS,
        "rare_row_share": rare["support"].sum() / N_ROWS,
        "eda_handling": (
            "Retain for EDA; summarize adequately supported levels "
            "and rare tail separately. No automatic deletion."
        ),
    })

high_cardinality = pd.DataFrame(rows)

high_cardinality.to_csv(
    OUTPUT_DIR / "high_cardinality_evidence.csv",
    index=False
)

display(high_cardinality)

,field,total_levels,supported_levels,rare_levels,rare_rows,denominator,rare_row_share,eda_handling
0,cat116,326,103,223,6673,188318,0.035435,Retain for EDA; summarize adequately supported...
1,cat110,131,39,92,2916,188318,0.015484,Retain for EDA; summarize adequately supported...
2,cat109,84,13,71,2791,188318,0.014821,Retain for EDA; summarize adequately supported...


## 6. Verify Every Target Comparison Uses Raw-Unit Evidence

Every categorical level-to-target comparison must include support and raw target summaries.

`log1p(loss)` may be used for readable plots, but it cannot replace the raw-unit evidence.

In [9]:
required_target_evidence = [
    "support",
    "mean_loss",
    "median_loss",
    "q25_loss",
    "q75_loss",
    "iqr_loss",
]

assert level_tables[required_target_evidence].notna().all().all()

assert (level_tables["support"] > 0).all()

assert np.allclose(
    level_tables["iqr_loss"],
    level_tables["q75_loss"] - level_tables["q25_loss"]
)

print("Categorical target evidence: PASS")
print("Every observed field-level comparison includes:")
print("- support")
print("- raw mean loss")
print("- raw median loss")
print("- raw loss quartiles")
print("- raw loss IQR")

Categorical target evidence: PASS
Every observed field-level comparison includes:
- support
- raw mean loss
- raw median loss
- raw loss quartiles
- raw loss IQR


## 7. Final Issue #20 Verification

Confirm that every requirement in the issue is supported by the frozen evidence and Gate 3 outputs.

In [10]:
checks = {
    "inventory_covers_116_fields":
        inventory["field"].nunique() == 116,

    "required_starting_checks_reproduced":
        (
            (inventory["n_levels"] == 2).sum() == 72
            and (inventory["n_levels"] <= 4).sum() == 88
            and inventory.loc[
                inventory["field"] == "cat116",
                "n_levels"
            ].iloc[0] == 326
            and inventory.loc[
                inventory["field"] == "cat110",
                "n_levels"
            ].iloc[0] == 131
            and inventory.loc[
                inventory["field"] == "cat109",
                "n_levels"
            ].iloc[0] == 84
        ),

    "low_support_evidence_created":
        len(low_support) > 0,

    "dominant_level_evidence_created":
        len(dominant) > 0,

    "uneven_target_evidence_created":
        len(uneven) > 0,

    "high_cardinality_fields_reviewed":
        set(high_cardinality["field"])
        == {"cat116", "cat110", "cat109"},

    "raw_target_evidence_complete":
        level_tables[
            required_target_evidence
        ].notna().all().all(),
}

verification = pd.DataFrame(
    [
        {
            "requirement": requirement,
            "status": "PASS" if passed else "FAIL",
        }
        for requirement, passed in checks.items()
    ]
)

display(verification)

assert all(checks.values())

print("\nIssue #20 evidence verification: PASS")

,requirement,status
0,inventory_covers_116_fields,PASS
1,required_starting_checks_reproduced,PASS
2,low_support_evidence_created,PASS
3,dominant_level_evidence_created,PASS
4,uneven_target_evidence_created,PASS
5,high_cardinality_fields_reviewed,PASS
6,raw_target_evidence_complete,PASS



Issue #20 evidence verification: PASS
